# 06 — Exploring the Last Headroom

**Where we started this milestone:** LB 0.94978 / CV 0.94990. **Where it ends: LB 0.95006 / CV 0.95009 — past the 0.95 target.**

This milestone ran on two tracks in parallel:

1. **A proper neural member** — torch MLP with categorical embeddings and a class-weighted loss (the two things sklearn's MLP couldn't give us in milestone 5). PyTorch has no wheels for this machine's Intel Mac + Python 3.13, so training runs in a `.venv-torch` side-environment (Python 3.12 + torch 2.2.2, the last Intel-mac build) via `src/train_nn.py`; this notebook consumes its artifacts.
2. **Competitor intelligence** — a systematic read of the six most-voted public kernels, extracting *score-evidenced* techniques rather than folklore.

## What the competitor scan found (the short version)

- **Why everyone hits ~0.950:** the competition target was reverse-engineered by the community — it is essentially a hand-made threshold rule (sleep < 6h, stress level, activity; a depth-4 tree reproduces it) scoring ~0.99 on the original source dataset. The synthetic labels are that rule + generation noise, which puts a hard ceiling just above 0.95. The published honest single-model frontier is OOF ≈ 0.9506.
- **The one verified feature edge:** *exact-value categorization + cross-fitted multiclass target encoding* — factorize numerics at their exact repeated values, target-encode. Ablated at **+0.0009 OOF** by its author — but crucially, feeding a **RealMLP**, not a tree.
- **Diversity is the binding constraint:** one kernel LB-verified that adding a fourth *GBM* to a GBM blend hurt (−0.00035, 99.5% agreement) while a neural member helps. Also: geometric (log-space) blending beat arithmetic by +0.00012 when member calibrations differ.
- **Validated against our own ledger:** their ablations independently found missingness flags carry zero signal (our E1) and that class-weighted training + a *fixed* prior correction over-correct by ~0.045 when stacked (our *tuned*-multiplier approach is the safe variant — the search walks the correction back).
- **Dead ends:** the original external dataset is wired into two kernels and disabled in both (rare for Playground). Public LB above ~0.951 is mostly shared-submission-file blending — the S6E6 precedent (public #1 dropped out of the private top 20) says honest-OOF gains are what survive.

## The neural member (`src/train_nn.py`)

Architecture (RealMLP-lite): per-category embeddings (dim 8, `missing` as its own token — E2's lesson), numerics clipped to fold-quantiles, median-imputed, standardized, plus missingness flags; 512→256 SiLU with dropout 0.15; **class-weighted cross-entropy** (the alignment lever, finally available since we control the loop); AdamW + cosine schedule; early stopping on fold-validation balanced accuracy with best-state restore. All preprocessing statistics fit per training fold. Fold contract enforced by loading `artifacts/fold_id.npy` directly.

**Result: OOF 0.94819** — vs sklearn-MLP's 0.93620 in milestone 5. The class-weighted loss and embeddings closed most of the quality gap to the trees. But blended with the three GBMs it earned weight 0.2 and moved the blend **zero** (0.94990 → 0.94990): at error correlation 0.87–0.93 it was still too tree-like. Diversity without *sufficient* decorrelation buys nothing — which set up the TE experiments.

In [1]:
import pandas as pd
import numpy as np
from itertools import product
from sklearn.metrics import balanced_accuracy_score

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
CLASSES = ['at-risk', 'fit', 'unhealthy']
y = train['health_condition'].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
A = '../artifacts/'

oof = {'lgbm': np.mean([np.load(f'{A}{n}_oof.npy')
                        for n in ('lgbm_tuned', 'lgbm_seed1', 'lgbm_seed2')], axis=0),
       'cat': np.load(A + 'cat_oof.npy'), 'hgb': np.load(A + 'hgb_oof.npy'),
       'nn': np.load(A + 'nn_oof.npy'), 'nn_te': np.load(A + 'nn_te_oof.npy'),
       'lgbm_te': np.load(A + 'lgbm_te_oof.npy')}
for n, o in oof.items():
    print(f"{n:8s} OOF {balanced_accuracy_score(y, o.argmax(1)):.5f}")

lgbm     OOF 0.94831
cat      OOF 0.94923


hgb      OOF 0.94956
nn       OOF 0.94819


nn_te    OOF 0.94831
lgbm_te  OOF 0.94451


## E6 / E7 — the target-encoding pair

The public recipe, tried on both function classes (scripts: `src/train_te_lgbm.py`, `src/train_nn_te.py`):

- **E6, TE → LightGBM: 0.94451 (Δ −0.0044). It hurts trees, badly.** A tree can already split "value ≤ 7.2" natively; 24 dense target-encoded columns are mostly *noise with target flavor*, and the booster spent its entire raised iteration budget fitting it. The public ablation never claimed trees — extrapolating it there was our mistake, cheaply caught.
- **E7, TE → NN: 0.94831 (Δ +0.00012 over plain NN).** Direction confirmed — the same features that cost the tree 4 points mildly *helped* the net, because an MLP has no native mechanism to isolate exact repeated values and the TE columns hand it that signal densely. But magnitude far below the +0.0009 of the real thing: RealMLP's periodic numeric embeddings, internal 8-model ensembling, and EMA are load-bearing, not decoration.

**The lesson that generalizes: features and function classes are a joint hypothesis.** "Does feature X help?" is ill-posed; "does X help *this model family*?" is the real question, and the same 24 columns moved our two families in opposite directions by two orders of magnitude.

In [2]:
err = pd.DataFrame({n: (o.argmax(1) != y).astype(int) for n, o in oof.items()})
err.corr().round(3)

,lgbm,cat,hgb,nn,nn_te,lgbm_te
lgbm,1.000,0.902,0.906,0.866,0.876,0.881
cat,0.902,1.000,0.955,0.932,0.919,0.834
hgb,0.906,0.955,1.000,0.921,0.916,0.839
nn,0.866,0.932,0.921,1.000,0.912,0.802
nn_te,0.876,0.919,0.916,0.912,1.000,0.816
lgbm_te,0.881,0.834,0.839,0.802,0.816,1.000


**But look at the correlation matrix** — the two "failed" models are the two most decorrelated members we own (`lgbm_te` 0.80–0.88, `nn_te` 0.80 vs `nn`'s 0.87–0.93). E6 lost 4 points of accuracy but bought a genuinely different error pattern; E7's TE nudged the NN further from the trees. Whether that diversity outweighs the quality gap is not decidable by argument — the blend search decides.

## The final blend

Same machinery as milestones 4–5 (coarse→fine simplex grid on OOF, then per-class thresholds on the blended OOF), now over five members. Implementation note: at 5 members × 690k rows the naive loop got slow — probabilities are pre-stacked as one float32 tensor and balanced accuracy computed via `bincount`, a ~10× speedup that keeps the search interactive.

In [3]:
members = ['lgbm', 'cat', 'hgb', 'nn_te', 'lgbm_te']
P = np.stack([oof[n].astype('float32') for n in members])
cls_counts = np.bincount(y)

def bal_acc(pred):
    return float((np.bincount(y[pred == y], minlength=3) / cls_counts).mean())

def score_w(w):
    return bal_acc(np.tensordot(w.astype('float32'), P, 1).argmax(1))

best = (None, -1)
for ws in product(np.arange(0, 1.001, 0.1), repeat=4):
    if sum(ws) > 1.0001: continue
    w = np.array([*ws, 1 - sum(ws)]); s = score_w(w)
    if s > best[1]: best = (w, s)
w0, _ = best
for ws in product(*[np.clip(np.arange(w0[i]-0.1, w0[i]+0.101, 0.05), 0, 1) for i in range(4)]):
    if sum(ws) > 1.0001: continue
    w = np.array([*ws, 1 - sum(ws)]); s = score_w(w)
    if s > best[1]: best = (w, s)
w, s = best
pb = np.tensordot(w.astype('float32'), P, 1)

best_t = (1.0, 1.0, bal_acc(pb.argmax(1)))
for a in np.arange(0.5, 2.01, 0.05):
    for b in np.arange(0.5, 2.01, 0.05):
        st = bal_acc((pb * np.array([1, a, b], dtype='float32')).argmax(1))
        if st > best_t[2]: best_t = (a, b, st)
a0, b0, _ = best_t
for a in np.arange(a0-0.06, a0+0.06, 0.01):
    for b in np.arange(b0-0.06, b0+0.06, 0.01):
        st = bal_acc((pb * np.array([1, a, b], dtype='float32')).argmax(1))
        if st > best_t[2]: best_t = (a, b, st)
a, b, cv_final = best_t
print(f"weights: {dict(zip(members, w.round(2)))}")
print(f"argmax {s:.5f} -> + thresholds(fit={a:.2f}, unh={b:.2f}) = {cv_final:.5f}")

weights: {'lgbm': np.float64(0.1), 'cat': np.float64(0.1), 'hgb': np.float64(0.5), 'nn_te': np.float64(0.25), 'lgbm_te': np.float64(0.05)}
argmax 0.95005 -> + thresholds(fit=1.15, unh=1.15) = 0.95009


In [4]:
testp = {'lgbm': np.mean([np.load(f'{A}{n}_testp.npy')
                          for n in ('lgbm_tuned', 'lgbm_seed1', 'lgbm_seed2')], axis=0),
         'cat': np.load(A + 'cat_testp.npy'), 'hgb': np.load(A + 'hgb_testp.npy'),
         'nn_te': np.load(A + 'nn_te_testp.npy'), 'lgbm_te': np.load(A + 'lgbm_te_testp.npy')}
p = sum(w[i] * testp[n] for i, n in enumerate(members)) * np.array([1.0, a, b])
sub = pd.DataFrame({'id': test['id'],
                    'health_condition': [CLASSES[i] for i in p.argmax(1)]})
sub.to_csv('../submissions/06-nn-blend.csv', index=False)
print(sub['health_condition'].value_counts(normalize=True).round(4))

health_condition
at-risk      0.8093
unhealthy    0.1170
fit          0.0737
Name: proportion, dtype: float64


## Result

| | CV | public LB | CV − LB |
|---|---|---|---|
| milestone 5 best | 0.94990 | 0.94978 | +0.0001 |
| **milestone 6 blend** | **0.95009** | **0.95006** | +0.00003 |

**Past the 0.95 target, with the tightest CV↔LB agreement of the project.** The weight vector tells the story of the milestone: `hgb 0.5, nn_te 0.25, lgbm 0.1, cat 0.1, lgbm_te 0.05` — a quarter of the final answer comes from the neural member that didn't exist a day ago, and even the tree that TE "ruined" earns a sliver on pure decorrelation.

Both "failed" experiments are in the final blend. That's the closing lesson of the project: **in ensembling, the unit of value is not a model's score but its contribution to the committee** — and the only way to price that is to measure it, on honest out-of-fold predictions, against your own ledger.

Remaining distance to the public top (~0.953): per the competitor scan, largely shared-submission-file blending with documented private-split risk, plus a true RealMLP (~+0.0005 honest). We stop here — target met, on evidence we trust.